# Data Preparation and Cleaning

This notebook prepares the movie and book datasets for our capstone project: a generative
recommendation system that focuses on the **cold-start problem**.

Cold start happens when the system has little or no interaction history to learn from.

For our project this can mean:

- **New users** who have few or no ratings yet. This includes collaborative filtering that has nothing to go on.
- **New movies or books** that have few or no ratings. They never show up in similarity-based recommendations.
- In these cases the model has to fall back on **content information** (genres, descriptions,
  authors, directors, release years, etc.) to make reasonable recommendations.

The cleaning pipeline in this notebook will:

1. Load and inspect the raw movie, book, and interaction data.
2. Clean and standardize each dataset.
3. Build a unified item table (movies + books) with a combined text feature for content-based / generative models.
4. Create train / validation / test splits, plus dedicated **new-user** and **new-item** cold-start test sets.
5. Save everything as CSV files for the modeling team.

> **Note:** this workflow is split across two notebooks. This notebook covers loading,
> cleaning, and building the unified item / interaction tables (steps 1-3), and saves them
> to `../outputs/`. The train / validation / test and cold-start splits (steps 4-5) live in
> `train_test_split.ipynb` - run this notebook first.

### Import libraries

In [ ]:
# Library
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# make pandas output a bit easier to read
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

### Project paths and configuration

All paths are resolved relative to the project root (`netflix-cold-start/`), so the
notebook runs no matter what the kernel's current working directory is, and no absolute
paths are hard-coded. Every other tunable setting lives here too, so there is one place
to change them.

- raw movie inputs: `../data/raw/movies/`
- raw book inputs: `../data/raw/books/`
- generated outputs: `../outputs/` (created automatically)

In [ ]:
# ---- Project paths (portable - no hard-coded absolute paths) ----

def find_project_root(markers=("data", "notebooks", "outputs")):
    """Locate the project root folder regardless of the working directory.

    Walks upward from the notebook's own location (available in VS Code) or from
    the current working directory until it finds a folder that directly contains
    all of the `markers` subfolders.

    Parameters
    ----------
    markers : tuple of str
        Folder names that must all exist directly inside the project root.

    Returns
    -------
    pathlib.Path
        Absolute path of the project root.
    """
    starts = []
    nb_file = globals().get("__vsc_ipynb_file__")  # set by VS Code's notebook editor
    if nb_file:
        starts.append(Path(nb_file).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if all((candidate / m).is_dir() for m in markers):
                return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start the kernel inside the "
        "netflix-cold-start folder, or set PROJECT_ROOT manually."
    )

PROJECT_ROOT = find_project_root()
MOVIE_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "movies"
BOOK_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "books"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

# ---- Reproducibility ----
SEED = 42
np.random.seed(SEED)

# ---- Cleaning settings ----
# values that really mean "missing"
EMPTY_VALUES = ["", "unknown", "none", "nan", "n/a", "na"]
# publication years outside this range are treated as data errors
VALID_YEAR_RANGE = (1400, 2026)
# expected rating scale - update these bounds for your data
expected_min, expected_max = 1, 5

### Reusable function definitions

All reusable logic lives in the small functions below, one function per cell. Each takes
its inputs (dataframes, column names, settings) as parameters and returns its result, so
the same functions can be rerun later with different input files, columns, or settings.
The execution cells further down call them in the same order as the original workflow.

In [ ]:
def load_table(path, name, required=True):
    """Load one raw table, handling CSV and JSON-lines files (plain or gzipped).

    Parameters
    ----------
    path : str, pathlib.Path, or None
        File to load. ``.jsonl`` / ``.jsonl.gz`` files are read as JSON lines;
        everything else is read with ``pd.read_csv`` (which decompresses ``.gz`` itself).
    name : str
        Dataset name used in the printed messages (e.g. "movies").
    required : bool
        If False, a missing file (or ``path=None``) returns None instead of raising.

    Returns
    -------
    pandas.DataFrame or None
    """
    if path is None or not os.path.exists(path):
        if required:
            raise FileNotFoundError(f"{name}: file not found at {path}")
        print(f"No separate {name} file found - update {name}_path if this is wrong.")
        return None
    if str(path).endswith((".jsonl", ".jsonl.gz")):
        df = pd.read_json(path, lines=True)
    else:
        df = pd.read_csv(path)
    print(f"{name.capitalize()} loaded:", df.shape)
    return df

In [ ]:
def inspect(df, name):
    """Print a quick overview of one dataframe, for all three datasets.

    Shows shape, first rows, columns, dtypes, duplicate-row count, and missing values.

    Parameters
    ----------
    df : pandas.DataFrame
    name : str
        Heading printed above the overview.
    """
    print("=" * 60)
    print(name)
    print("=" * 60)
    print("Shape:", df.shape)
    print("\nFirst 5 rows:")
    display(df.head())
    print("Columns:", list(df.columns))
    print("\nData types:")
    print(df.dtypes)
    print("\nDuplicate rows:", df.duplicated().sum())
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    print("\nMissing values:")
    print(pd.DataFrame({"missing": missing, "missing_%": missing_pct}))

In [ ]:
def clean_columns(df):
    """Standardize column names: strip, lowercase, underscores for spaces/punctuation.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame
        A copy with cleaned column names.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[ \-/]+", "_", regex=True)
        .str.replace(r"[^0-9a-z_]", "", regex=True)
    )
    return df

In [ ]:
def check_cols(df, cols, name):
    """Warn about configured column names that do not exist in a dataframe.

    Simple existence checks so wrong names fail loudly, not silently.

    Parameters
    ----------
    df : pandas.DataFrame
    cols : list of (str or None)
        Column names to check; None entries are skipped.
    name : str
        Dataset name used in the warning message.
    """
    for c in cols:
        if c is not None and c not in df.columns:
            print(f"WARNING: '{c}' not found in {name} columns -> update the variable above")

In [ ]:
def drop_unusable_rows(df, id_col, title_col, name):
    """Remove exact duplicate rows and rows that are unusable for recommendation.

    Rows missing the item id or the title are dropped. Prints a before/after count.

    Parameters
    ----------
    df : pandas.DataFrame
    id_col, title_col : str
        Columns that must be non-missing for a row to be usable.
    name : str
        Dataset name used in the printed message (e.g. "Movies").

    Returns
    -------
    pandas.DataFrame
        A new dataframe without the removed rows.
    """
    before = len(df)

    # 1. exact duplicate rows
    df = df.drop_duplicates()

    # 2. rows missing the item id or title are unusable for recommendation
    df = df.dropna(subset=[id_col, title_col])

    print(f"{name}: {before} -> {len(df)} rows "
          f"({before - len(df)} removed)")
    return df

In [ ]:
def standardize_types_and_text(df, id_col, year_col, text_cols, empty_values=EMPTY_VALUES):
    """Fix column types and tidy the text fields of one item table.

    Ids become stripped strings (important for ISBNs - leading zeros matter and they
    are identifiers, not numbers), the year column becomes numeric, and text columns
    are stripped, with placeholder strings ("unknown", "n/a", ...) turned back into
    real missing values.

    Parameters
    ----------
    df : pandas.DataFrame
    id_col : str
        Identifier column, converted to clean strings.
    year_col : str
        Year column, converted to numeric (bad values become NaN).
    text_cols : list of str
        Text columns to tidy; columns not present are skipped.
    empty_values : list of str
        Lowercase strings that really mean "missing".

    Returns
    -------
    pandas.DataFrame
    """
    df = df.copy()
    df[id_col] = df[id_col].astype(str).str.strip()
    df[year_col] = pd.to_numeric(df[year_col], errors="coerce")

    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            # turn obvious empty strings back into real missing values
            df[col] = df[col].where(~df[col].str.lower().isin(empty_values), np.nan)
    return df

In [ ]:
def nullify_unrealistic_years(df, year_col, valid_range, name):
    """Treat years outside `valid_range` as missing, without deleting any rows.

    We look at the suspicious values BEFORE deciding what to do about them: the
    counts and most common bad values are printed. The rows themselves are kept -
    the item is still valid, we just don't trust the year.

    Parameters
    ----------
    df : pandas.DataFrame
    year_col : str
    valid_range : tuple of (int, int)
        Inclusive (min_year, max_year) range considered realistic.
    name : str
        Dataset name used in the printed messages (e.g. "Books").

    Returns
    -------
    pandas.DataFrame
    """
    df = df.copy()
    year_ok = df[year_col].between(*valid_range)
    weird_years = df.loc[~year_ok & df[year_col].notna(), year_col]

    print(f"{name} with unrealistic {year_col.replace('_', ' ')} values: {len(weird_years)} "
          f"({len(weird_years) / len(df) * 100:.2f}% of {name.lower()})")
    print(weird_years.value_counts().head(10))

    # we don't delete these rows - the item itself is still valid,
    # we just treat the bad year as missing
    df.loc[~year_ok, year_col] = np.nan
    return df

In [ ]:
def fill_missing_text(df, cols, name):
    """Fill missing values in content/text columns with "" and report what is left.

    Filling with "" makes later text concatenation painless. Numeric columns like
    release_year should NOT be passed here - filling them with a fake number would
    be misleading.

    Parameters
    ----------
    df : pandas.DataFrame
    cols : list of str
        Text columns to fill; columns not present are skipped.
    name : str
        Dataset name used in the printed message.

    Returns
    -------
    pandas.DataFrame
    """
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].fillna("")

    print(f"Remaining missing values in {name}:")
    print(df.isna().sum()[df.isna().sum() > 0])
    return df

In [ ]:
def clean_interactions(df, user_col, item_col, rating_col):
    """Clean a raw user-item ratings table.

    Drops rows missing user or item ids, converts ids to strings so they match the
    item tables later, converts ratings to numeric (dropping unparseable ones), and
    removes duplicate user-item-rating records. Prints a before/after count.

    Parameters
    ----------
    df : pandas.DataFrame
    user_col, item_col, rating_col : str

    Returns
    -------
    pandas.DataFrame
    """
    before = len(df)

    # rows missing user or item id are useless
    df = df.dropna(subset=[user_col, item_col]).copy()

    # IDs as strings so they match the item tables later
    df[user_col] = df[user_col].astype(str).str.strip()
    df[item_col] = df[item_col].astype(str).str.strip()

    # ratings to numeric
    df[rating_col] = pd.to_numeric(df[rating_col], errors="coerce")
    df = df.dropna(subset=[rating_col])

    # duplicate user-item-rating records
    df = df.drop_duplicates(subset=[user_col, item_col, rating_col])

    print(f"Ratings: {before} -> {len(df)} rows "
          f"({before - len(df)} removed)")
    return df

In [ ]:
def pct_empty(series):
    """Return the percentage of values in `series` that are missing or empty text.

    Parameters
    ----------
    series : pandas.Series

    Returns
    -------
    float
        Percentage between 0 and 100.
    """
    return (series.fillna("").astype(str).str.strip() == "").mean() * 100

### Load the datasets

We will be evaluating Netflix's Movie and Book dataset and will load the following three files:

- the **movie** dataset (Netflix-related, ~2023)
- the **book** dataset
- the **user–item interaction / ratings** data (if stored separately)

> **Note (project reorganization, 2026-07-13):** the raw files in `data/raw/` are from the
> **Amazon Reviews 2023** dataset (see `README.md`), which the team docs confirm is this
> project's data despite the "Netflix" name above: movie metadata
> (`meta_Movies_and_TV.jsonl.gz`), movie ratings (`Movies_and_TV.csv.gz`), and book ratings
> (`Books.csv.gz`). There is no separate book *catalog* file - `Books.csv.gz` contains book
> ratings. The paths below are built from the project folders set up in the configuration cell.

In [ ]:
# Real file locations, relative to the project folders defined in the configuration cell.
# NOTE: the movie *catalog* is the metadata file; both ratings files are separate.
movie_path = MOVIE_DATA_DIR / "meta_Movies_and_TV.jsonl.gz"   # movie catalog / metadata
book_path = BOOK_DATA_DIR / "Books.csv.gz"                    # book ratings (no separate book catalog exists)
ratings_path = MOVIE_DATA_DIR / "Movies_and_TV.csv.gz"        # movie ratings; set to None if interactions are inside the item files

In [ ]:
# load the movie dataset (catalog / metadata)
movies_raw = load_table(movie_path, "movies")

In [ ]:
# load the book dataset
books_raw = load_table(book_path, "books")

In [ ]:
# load the ratings / interactions dataset (skip if it does not exist as a separate file)
ratings_raw = load_table(ratings_path, "ratings", required=False)

### Inspect data

In [ ]:
inspect(movies_raw, "MOVIES")

In [ ]:
inspect(books_raw, "BOOKS")

In [ ]:
if ratings_raw is not None:
    inspect(ratings_raw, "RATINGS / INTERACTIONS")

In [ ]:
# small summary table of the raw datasets
summary_rows = [
    {"dataset": "movies", "rows": movies_raw.shape[0], "columns": movies_raw.shape[1],
     "duplicate_rows": movies_raw.duplicated().sum()},
    {"dataset": "books", "rows": books_raw.shape[0], "columns": books_raw.shape[1],
     "duplicate_rows": books_raw.duplicated().sum()},
]
if ratings_raw is not None:
    summary_rows.append({"dataset": "ratings", "rows": ratings_raw.shape[0],
                         "columns": ratings_raw.shape[1],
                         "duplicate_rows": ratings_raw.duplicated().sum()})

pd.DataFrame(summary_rows)

**Observations (edit after running):**

- Movies: *e.g., X rows, Y columns, notable missing values in ...*
- Books: *...*
- Ratings: *...*


## 5. Standardize column names

Lowercase, strip spaces, and replace spaces/punctuation with underscores so the rest of
the notebook can use consistent names.

In [ ]:
movies = clean_columns(movies_raw)
books = clean_columns(books_raw)
ratings = clean_columns(ratings_raw) if ratings_raw is not None else None

print("Movie columns: ", list(movies.columns))
print("Book columns:  ", list(books.columns))
if ratings is not None:
    print("Rating columns:", list(ratings.columns))

## 6. Identify important columns

The exact column names depend on the files we end up using, so instead of guessing
silently, we set them as variables in one place. **Update these after checking the
printed column lists above.**

> **Note (project reorganization, 2026-07-13):** the placeholders below describe the
> originally planned Netflix-style schema and do **not** match the Amazon Reviews 2023
> files now in `data/raw/`. Columns actually found in the raw files:
>
> - movie metadata (`meta_Movies_and_TV.jsonl.gz`): `main_category`, `title`, `subtitle`,
>   `average_rating`, `rating_number`, `features`, `description`, `price`, `images`,
>   `videos`, `store`, `categories`, `details`, `parent_asin`, `bought_together`
> - movie ratings and book ratings files: `user_id`, `parent_asin`, `rating`, `timestamp`
>
> The check cell below will print warnings until these variables are updated. Choosing
> the new mappings is a team decision (see README), so they are left as-is on purpose.

In [ ]:
# ---- MOVIE dataset columns (update to match your file) ----
movie_id_col = "show_id"        # unique movie identifier
movie_title_col = "title"
movie_genre_col = "listed_in"   # Netflix datasets often call genres "listed_in"
movie_desc_col = "description"
movie_director_col = "director"
movie_cast_col = "cast"
movie_year_col = "release_year"

# ---- BOOK dataset columns (update to match your file) ----
book_id_col = "isbn"            # or "book_id"
book_title_col = "title"
book_author_col = "author"      # or "authors"
book_genre_col = "genre"        # or "categories"
book_desc_col = "description"
book_year_col = "publication_year"

# ---- RATINGS dataset columns (update to match your file) ----
user_col = "user_id"
rating_item_col = "item_id"     # the raw item id in the ratings file
rating_col = "rating"
timestamp_col = "timestamp"     # set to None if there is no timestamp

In [ ]:
# simple existence checks so wrong names fail loudly, not silently
check_cols(movies, [movie_id_col, movie_title_col, movie_genre_col, movie_desc_col,
                    movie_director_col, movie_cast_col, movie_year_col], "movies")
check_cols(books, [book_id_col, book_title_col, book_author_col, book_genre_col,
                   book_desc_col, book_year_col], "books")
if ratings is not None:
    check_cols(ratings, [user_col, rating_item_col, rating_col, timestamp_col], "ratings")
print("Column check finished (no output above the line = all good).")

## 7. Clean the movie dataset

Simple, conservative cleaning. We remove exact duplicates and rows that are unusable
(no ID or no title), fix types, and tidy the text fields. We deliberately avoid
aggressive cleaning that could throw away information the models might need.

In [ ]:
movies = drop_unusable_rows(movies, movie_id_col, movie_title_col, "Movies")

In [ ]:
# fix types and tidy text
text_cols_movies = [movie_title_col, movie_genre_col, movie_desc_col,
                    movie_director_col, movie_cast_col]

movies = standardize_types_and_text(movies, movie_id_col, movie_year_col,
                                    text_cols_movies, EMPTY_VALUES)

movies[[movie_id_col, movie_title_col, movie_year_col]].head()

In [ ]:
# fill missing content fields with "" so text concatenation later is painless
# (we keep release_year as NaN - filling it with a fake number would be misleading)
movies = fill_missing_text(movies, [movie_genre_col, movie_desc_col,
                                    movie_director_col, movie_cast_col], "movies")

## 8. Clean the book dataset

Same idea as the movies: duplicates, unusable rows, types, and text tidying.
ISBNs are kept as **strings** because leading zeros matter and they are identifiers,
not numbers.

In [ ]:
books = drop_unusable_rows(books, book_id_col, book_title_col, "Books")

In [ ]:
# IDs as clean strings (important for ISBNs), publication year to numeric, tidy text
text_cols_books = [book_title_col, book_author_col, book_genre_col, book_desc_col]

books = standardize_types_and_text(books, book_id_col, book_year_col,
                                   text_cols_books, EMPTY_VALUES)

books[[book_id_col, book_title_col, book_year_col]].head()

In [ ]:
# look at suspicious publication years BEFORE deciding what to do about them;
# bad years become missing, the rows themselves are kept
books = nullify_unrealistic_years(books, book_year_col, VALID_YEAR_RANGE, "Books")

In [ ]:
# fill missing text content with "" like we did for movies
books = fill_missing_text(books, [book_author_col, book_genre_col, book_desc_col], "books")

## 9. Clean the interaction / rating data

Interactions are the core input for collaborative filtering, and how sparse they are
is exactly what makes cold start hard. We clean carefully and **look before we delete**.

In [ ]:
if ratings is not None:
    ratings = clean_interactions(ratings, user_col, rating_item_col, rating_col)

In [ ]:
if ratings is not None:
    print("Rating value counts:")
    print(ratings[rating_col].value_counts().sort_index())
    print("\nMin rating:", ratings[rating_col].min())
    print("Max rating:", ratings[rating_col].max())

    # check for ratings outside the expected scale (bounds set in the configuration cell)
    out_of_scale = ratings[(ratings[rating_col] < expected_min) |
                           (ratings[rating_col] > expected_max)]
    print(f"\nRatings outside [{expected_min}, {expected_max}]: {len(out_of_scale)}")
    if len(out_of_scale) > 0:
        display(out_of_scale.head())
        # decide manually whether to drop these - we do NOT drop them automatically

In [ ]:
if ratings is not None and timestamp_col is not None and timestamp_col in ratings.columns:
    # many datasets store unix seconds; pd.to_datetime handles both cases with errors="coerce"
    ts = ratings[timestamp_col]
    if pd.api.types.is_numeric_dtype(ts):
        ratings[timestamp_col] = pd.to_datetime(ts, unit="s", errors="coerce")
    else:
        ratings[timestamp_col] = pd.to_datetime(ts, errors="coerce")

    ratings = ratings.sort_values(timestamp_col).reset_index(drop=True)
    print("Timestamp range:", ratings[timestamp_col].min(), "->", ratings[timestamp_col].max())
    HAS_TIMESTAMP = ratings[timestamp_col].notna().mean() > 0.9  # mostly valid timestamps
else:
    HAS_TIMESTAMP = False

print("Using timestamps for splitting:", HAS_TIMESTAMP)

## 10. Basic exploratory analysis

A few simple plots and numbers. The main thing we care about here is **sparsity**:
how many users and items have very few interactions, because those are exactly the
cold-start cases our system needs to handle.

In [ ]:
if ratings is not None:
    plt.figure(figsize=(6, 4))
    ratings[rating_col].hist(bins=20)
    plt.title("Rating distribution")
    plt.xlabel("Rating")
    plt.ylabel("Count")
    plt.show()

In [ ]:
if ratings is not None:
    interactions_per_user = ratings.groupby(user_col).size()
    interactions_per_item = ratings.groupby(rating_item_col).size()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(interactions_per_user, bins=50)
    axes[0].set_title("Interactions per user")
    axes[0].set_xlabel("Number of interactions")
    axes[0].set_ylabel("Users")

    axes[1].hist(interactions_per_item, bins=50)
    axes[1].set_title("Interactions per item")
    axes[1].set_xlabel("Number of interactions")
    axes[1].set_ylabel("Items")
    plt.tight_layout()
    plt.show()

In [ ]:
if ratings is not None:
    print("Most frequently rated items (raw ids):")
    print(interactions_per_item.sort_values(ascending=False).head(10))

    few_user = (interactions_per_user <= 5).mean() * 100
    few_item = (interactions_per_item <= 5).mean() * 100
    print(f"\nUsers with <= 5 interactions: {few_user:.1f}%")
    print(f"Items with <= 5 interactions: {few_item:.1f}%")

In [ ]:
# how much content information is missing? (matters for content-based fallback)
print("Movies - % empty content fields:")
for col in [movie_genre_col, movie_desc_col, movie_director_col]:
    if col in movies.columns:
        print(f"  {col}: {pct_empty(movies[col]):.1f}%")

print("Books - % empty content fields:")
for col in [book_genre_col, book_desc_col, book_author_col]:
    if col in books.columns:
        print(f"  {col}: {pct_empty(books[col]):.1f}%")

**What this tells us about cold start (edit after running):**

- If a large share of users/items have ≤5 interactions, the "long tail" is big and
  pure collaborative filtering will struggle for most of the catalog.
- The most-rated items show how concentrated the interactions are on popular items.
- Missing content fields matter because content is our fallback for cold-start items —
  an item with no description *and* no interactions is the hardest possible case.


## 11. Create a unified item table

We combine movies and books into one item table with standardized columns.
IDs get a `movie_` / `book_` prefix so they can never collide. Columns that only
exist in one dataset are simply created as blanks in the other before concatenating.

In [ ]:
# standardized movie items
movie_items = pd.DataFrame({
    "item_id": "movie_" + movies[movie_id_col].astype(str),
    "item_type": "movie",
    "title": movies[movie_title_col],
    "creator": movies[movie_director_col] if movie_director_col in movies.columns else "",
    "genre": movies[movie_genre_col] if movie_genre_col in movies.columns else "",
    "description": movies[movie_desc_col] if movie_desc_col in movies.columns else "",
    "release_year": movies[movie_year_col] if movie_year_col in movies.columns else np.nan,
})

# standardized book items
book_items = pd.DataFrame({
    "item_id": "book_" + books[book_id_col].astype(str),
    "item_type": "book",
    "title": books[book_title_col],
    "creator": books[book_author_col] if book_author_col in books.columns else "",
    "genre": books[book_genre_col] if book_genre_col in books.columns else "",
    "description": books[book_desc_col] if book_desc_col in books.columns else "",
    "release_year": books[book_year_col] if book_year_col in books.columns else np.nan,
})

items = pd.concat([movie_items, book_items], ignore_index=True)
items = items.drop_duplicates(subset=["item_id"])

print("Unified item table:", items.shape)
print(items["item_type"].value_counts())
items.head(8)

## 12. Create a combined text feature

We build one `content_text` column per item by joining the title, type, genre,
creator (author/director), and description.

This text is the **content representation** of each item. Later, a generative model,
an embedding model, or a simple TF-IDF content-based recommender can use it to
recommend brand-new items that have zero interactions — which is the whole point
of the new-item cold-start setup. We are **not** generating embeddings here.

In [ ]:
items["content_text"] = (
    items["title"].fillna("") + " " +
    items["item_type"].fillna("") + " " +
    items["genre"].fillna("") + " " +
    items["creator"].fillna("") + " " +
    items["description"].fillna("")
)

# tidy up: collapse repeated spaces and lowercase
items["content_text"] = (
    items["content_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

items[["item_id", "content_text"]].head()

## 13. Prepare interaction data for modeling

The ratings file uses raw IDs, so we map them onto the prefixed IDs from the unified
item table. Because a raw ID could belong to either dataset, we check membership in
the movie IDs first, then the book IDs.

We do **not** silently drop interactions that don't match an item — we count them first.

In [ ]:
if ratings is not None:
    movie_ids = set(movies[movie_id_col].astype(str))
    book_ids = set(books[book_id_col].astype(str))

    def to_prefixed_id(raw_id):
        if raw_id in movie_ids:
            return "movie_" + raw_id
        if raw_id in book_ids:
            return "book_" + raw_id
        return None  # no matching item

    interactions = ratings.copy()
    interactions["item_id"] = interactions[rating_item_col].map(to_prefixed_id)

    unmatched = interactions["item_id"].isna().sum()
    print(f"Interactions that match no item: {unmatched} "
          f"({unmatched / len(interactions) * 100:.2f}%)")

    # NOTE: if your ratings file already has an item_type column or separate
    # movie/book rating files, prefix the ids directly instead of using the lookup above.

In [ ]:
if ratings is not None:
    # keep only matched interactions and add item_type
    interactions = interactions.dropna(subset=["item_id"]).copy()
    interactions = interactions.merge(items[["item_id", "item_type"]], on="item_id", how="left")

    keep_cols = [user_col, "item_id", rating_col, "item_type"]
    if HAS_TIMESTAMP:
        keep_cols.insert(3, timestamp_col)
    interactions = interactions[keep_cols].rename(
        columns={user_col: "user_id", rating_col: "rating", timestamp_col: "timestamp"}
        if HAS_TIMESTAMP else {user_col: "user_id", rating_col: "rating"})

    print("Clean interaction table:", interactions.shape)
    interactions.head()

## Save the cleaned tables

Everything the split notebook needs, saved to the project `outputs/` folder
(`../outputs/` relative to this notebook). `train_test_split.ipynb` picks up from here.

In [ ]:
# save the cleaned tables for train_test_split.ipynb
out_dir = OUTPUT_DIR   # project outputs folder, set in the configuration cell
os.makedirs(out_dir, exist_ok=True)

items.to_csv(os.path.join(out_dir, "processed_items.csv"), index=False)

if ratings is not None:
    interactions.to_csv(os.path.join(out_dir, "processed_interactions.csv"), index=False)

print("Saved files:")
for f in sorted(os.listdir(out_dir)):
    print(" -", f)